In [3]:
!pip install "numpy==2.2.0" --force-reinstall -q
!pip install nuscenes-devkit -q
print('Done.')

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
nuscenes-devkit 1.2.0 requires numpy<2.0.0,>=1.22.0, but you have numpy 2.2.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
spopt 0.7.0 requires shapely>=2.1.0, but you have shapely 2.0.7 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.2.0 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.0 which is incompatible.
pointpats 2.5.5 requires shapely>=2.1, but you have shapely 2.0.7 w

In [4]:
import os, shutil

# Clean up if needed
out = '/kaggle/working/nuscenes/'
for folder in ['samples', 'sweeps']:
    path = os.path.join(out, folder)
    if os.path.exists(path):
        shutil.rmtree(path)

from nuscenes.nuscenes import NuScenes
nusc = NuScenes(
    version  = 'v1.0-trainval',
    dataroot = '/kaggle/working/nuscenes/',
    verbose  = False
)
print(f'Scenes      : {len(nusc.scene)}')
print(f'Samples     : {len(nusc.sample)}')
print(f'Annotations : {len(nusc.sample_annotation)}')
print(f'Instances   : {len(nusc.instance)}')

Scenes      : 850
Samples     : 34149
Annotations : 1166187
Instances   : 64386


In [5]:
import os
from dataclasses import dataclass

@dataclass
class CFG:
    DATAROOT     : str   = '/kaggle/working/nuscenes/'
    VERSION      : str   = 'v1.0-trainval'
    SAVE_DIR     : str   = '/kaggle/working/checkpoints'
    # Trajectory settings
    OBS_SECS     : float = 2.0   # observe 2 seconds of past
    PRED_SECS    : float = 3.0   # predict 3 seconds future
    FREQ         : float = 2.0   # nuScenes annotation frequency (2Hz)
    OBS_LEN      : int   = 4     # 2s x 2Hz = 4 past positions
    PRED_LEN     : int   = 6     # 3s x 2Hz = 6 future positions
    NUM_MODES    : int   = 3     # predict 3 possible paths
    # Model
    D_MODEL      : int   = 128
    N_HEADS      : int   = 8
    N_LAYERS     : int   = 4
    DROPOUT      : float = 0.1
    MAX_AGENTS   : int   = 20    # max neighbours per scene
    # Training
    EPOCHS       : int   = 60
    BATCH        : int   = 64
    LR           : float = 1e-3
    WD           : float = 1e-4
    SEED         : int   = 42
    DEVICE       : str   = 'cuda'
    NUM_WORKERS  : int   = 2

cfg = CFG()
os.makedirs(cfg.SAVE_DIR, exist_ok=True)
print('Config loaded.')
print(f'Obs steps : {cfg.OBS_LEN} | Pred steps : {cfg.PRED_LEN}')

Config loaded.
Obs steps : 4 | Pred steps : 6


In [6]:
import random, numpy as np, torch

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

seed_everything(cfg.SEED)
print(f'Seeds fixed to {cfg.SEED}')

Seeds fixed to 42


In [7]:
import numpy as np
from nuscenes.nuscenes import NuScenes

def extract_trajectories(nusc, obs_len=4, pred_len=6):
    '''
    Extract pedestrian and cyclist trajectories from nuScenes.
    Returns list of dicts with obs (past) and pred (future) positions.
    Categories: pedestrian, bicycle, motorcycle
    '''
    TARGET_CATS = {'human.pedestrian.adult',
                   'human.pedestrian.child',
                   'human.pedestrian.wheelchair',
                   'vehicle.bicycle',
                   'vehicle.motorcycle'}

    # Build per-instance timeline: instance_token -> list of (sample_idx, x, y)
    print('Building instance timelines...')
    instance_timeline = {}

    for ann in nusc.sample_annotation:
        inst_token = ann['instance_token']
        cat = nusc.get('category',
              nusc.get('instance', inst_token)['category_token'])['name']
        if not any(cat.startswith(t) for t in TARGET_CATS):
            continue
        sample = nusc.get('sample', ann['sample_token'])
        x, y   = ann['translation'][0], ann['translation'][1]
        if inst_token not in instance_timeline:
            instance_timeline[inst_token] = []
        instance_timeline[inst_token].append(
            (sample['timestamp'], x, y, cat))

    # Sort each instance by timestamp
    for k in instance_timeline:
        instance_timeline[k].sort(key=lambda t: t[0])

    # Slide window: obs_len past + pred_len future
    print('Extracting sequences...')
    sequences = []
    total_len = obs_len + pred_len

    for inst_token, timeline in instance_timeline.items():
        if len(timeline) < total_len:
            continue
        for i in range(len(timeline) - total_len + 1):
            window = timeline[i:i+total_len]
            obs  = np.array([[w[1], w[2]] for w in window[:obs_len]],
                            dtype=np.float32)
            pred = np.array([[w[1], w[2]] for w in window[obs_len:]],
                            dtype=np.float32)
            sequences.append({
                'obs':       obs,       # (obs_len, 2)
                'pred':      pred,      # (pred_len, 2)
                'category':  window[0][3],
                'inst_token': inst_token
            })

    print(f'Total sequences: {len(sequences)}')
    return sequences

sequences = extract_trajectories(nusc, cfg.OBS_LEN, cfg.PRED_LEN)
print(f'Sample obs  shape: {sequences[0]["obs"].shape}')
print(f'Sample pred shape: {sequences[0]["pred"].shape}')

Building instance timelines...
Extracting sequences...
Total sequences: 136078
Sample obs  shape: (4, 2)
Sample pred shape: (6, 2)


In [8]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import numpy as np

class TrajectoryDataset(Dataset):
    def __init__(self, sequences, max_agents=20):
        self.sequences  = sequences
        self.max_agents = max_agents
        # Group sequences by instance for social context
        self.inst_map = {}
        for i, s in enumerate(sequences):
            self.inst_map.setdefault(s['inst_token'], []).append(i)

    def __len__(self): return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        obs  = torch.tensor(seq['obs'],  dtype=torch.float32)  # (obs_len,2)
        pred = torch.tensor(seq['pred'], dtype=torch.float32)  # (pred_len,2)

        # Normalise: centre on last observed position
        origin = obs[-1].clone()
        obs    = obs  - origin
        pred   = pred - origin

        return obs, pred

# Split 80/10/10
n       = len(sequences)
n_train = int(0.80 * n)
n_val   = int(0.10 * n)
n_test  = n - n_train - n_val

train_ds, val_ds, test_ds = random_split(
    TrajectoryDataset(sequences),
    [n_train, n_val, n_test],
    generator=torch.Generator().manual_seed(cfg.SEED)
)

train_loader = DataLoader(train_ds, batch_size=cfg.BATCH,
    shuffle=True,  num_workers=cfg.NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=cfg.BATCH,
    shuffle=False, num_workers=cfg.NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=cfg.BATCH,
    shuffle=False, num_workers=cfg.NUM_WORKERS, pin_memory=True)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
print('DataLoaders ready.')

Train: 108862 | Val: 13607 | Test: 13609
DataLoaders ready.


In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=100, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

class SocialTransformer(nn.Module):
    '''
    Social Transformer for multi-modal trajectory prediction.
    Input  : obs trajectory (obs_len, 2)
    Output : K predicted trajectories (num_modes, pred_len, 2)
             + K confidence scores
    '''
    def __init__(self, obs_len=4, pred_len=6, d_model=128,
                 n_heads=8, n_layers=4, num_modes=3, dropout=0.1):
        super().__init__()
        self.obs_len   = obs_len
        self.pred_len  = pred_len
        self.d_model   = d_model
        self.num_modes = num_modes

        # Input embedding
        self.input_embed = nn.Sequential(
            nn.Linear(2, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, d_model)
        )

        # Velocity embedding
        self.vel_embed = nn.Sequential(
            nn.Linear(2, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, d_model)
        )

        # Positional encoding
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout)

        # Transformer encoder (social context)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer,
                                                  num_layers=n_layers)

        # Multi-modal decoder heads
        self.mode_queries = nn.Parameter(
            torch.randn(num_modes, d_model))

        dec_layer = nn.TransformerDecoderLayer(
            d_model=d_model, nhead=n_heads,
            dim_feedforward=d_model*4,
            dropout=dropout, batch_first=True
        )
        self.decoder = nn.TransformerDecoder(dec_layer,
                                              num_layers=n_layers)

        # Output heads
        self.traj_head = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, pred_len * 2)
        )
        self.conf_head = nn.Sequential(
            nn.Linear(d_model, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, obs):
        '''
        obs: (B, obs_len, 2)
        returns:
            trajs: (B, num_modes, pred_len, 2)
            confs: (B, num_modes)
        '''
        B = obs.size(0)

        # Compute velocities
        vel = torch.zeros_like(obs)
        vel[:, 1:] = obs[:, 1:] - obs[:, :-1]

        # Embed positions + velocities
        pos_emb = self.input_embed(obs)   # (B, obs_len, d)
        vel_emb = self.vel_embed(vel)     # (B, obs_len, d)
        x = pos_emb + vel_emb

        # Add positional encoding
        x = self.pos_enc(x)

        # Encode with transformer
        memory = self.transformer(x)     # (B, obs_len, d)

        # Multi-modal decoding
        queries = self.mode_queries.unsqueeze(0).expand(B, -1, -1)
        # (B, num_modes, d)

        decoded = self.decoder(queries, memory)  # (B, num_modes, d)

        # Predict trajectories and confidences
        trajs = self.traj_head(decoded)          # (B, num_modes, pred_len*2)
        trajs = trajs.view(B, self.num_modes,
                           self.pred_len, 2)     # (B, num_modes, pred_len, 2)
        confs = self.conf_head(decoded).squeeze(-1)  # (B, num_modes)
        confs = F.softmax(confs, dim=-1)

        return trajs, confs

model = SocialTransformer(
    obs_len   = cfg.OBS_LEN,
    pred_len  = cfg.PRED_LEN,
    d_model   = cfg.D_MODEL,
    n_heads   = cfg.N_HEADS,
    n_layers  = cfg.N_LAYERS,
    num_modes = cfg.NUM_MODES,
    dropout   = cfg.DROPOUT
).to(cfg.DEVICE)

total = sum(p.numel() for p in model.parameters()) / 1e6
print(f'Model parameters: {total:.2f}M')
print('Social Transformer ready.')

Model parameters: 1.88M
Social Transformer ready.


In [10]:
import torch

def compute_ade(pred_trajs, gt_traj):
    '''
    pred_trajs : (B, num_modes, pred_len, 2)
    gt_traj    : (B, pred_len, 2)
    Returns best-of-K ADE (lower is better)
    '''
    gt = gt_traj.unsqueeze(1).expand_as(pred_trajs)
    dist = torch.norm(pred_trajs - gt, dim=-1)  # (B, K, pred_len)
    ade_per_mode = dist.mean(dim=-1)             # (B, K)
    best_ade     = ade_per_mode.min(dim=-1)[0]   # (B,)
    return best_ade.mean().item()

def compute_fde(pred_trajs, gt_traj):
    '''
    Returns best-of-K FDE (lower is better)
    '''
    gt_final = gt_traj[:, -1, :]                 # (B, 2)
    pred_final = pred_trajs[:, :, -1, :]         # (B, K, 2)
    dist = torch.norm(pred_final -
           gt_final.unsqueeze(1), dim=-1)        # (B, K)
    best_fde = dist.min(dim=-1)[0]               # (B,)
    return best_fde.mean().item()

print('ADE/FDE metrics ready.')

ADE/FDE metrics ready.


In [13]:
import time, os
import torch.nn as nn
import torch.nn.functional as F
from torch.amp import GradScaler, autocast

# Reload best checkpoint to continue from there
ckpt = torch.load(os.path.join(cfg.SAVE_DIR, 'model_best.pth'))
model.load_state_dict(ckpt['state_dict'])
start_epoch = ckpt['epoch'] + 1
best_ade    = ckpt['ade']
print(f'Resuming from epoch {start_epoch}, ADE={best_ade:.4f}')

# Lower LR + no AMP to prevent NaN
optimizer = torch.optim.AdamW(
    model.parameters(), lr=1e-4, weight_decay=cfg.WD)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=cfg.EPOCHS - start_epoch + 1, eta_min=1e-6)

criterion_traj = nn.SmoothL1Loss()

for epoch in range(start_epoch, cfg.EPOCHS+1):
    model.train()
    train_loss = 0.0
    t0 = time.time()
    skip_count = 0

    for obs, pred in train_loader:
        obs  = obs.to(cfg.DEVICE)
        pred = pred.to(cfg.DEVICE)

        optimizer.zero_grad()
        trajs, confs = model(obs)

        gt_exp    = pred.unsqueeze(1).expand_as(trajs)
        traj_loss = criterion_traj(trajs, gt_exp)

        with torch.no_grad():
            dist      = torch.norm(trajs - gt_exp, dim=-1).mean(dim=-1)
            best_mode = dist.argmin(dim=-1)
        conf_loss = F.nll_loss(torch.log(confs + 1e-8), best_mode)
        loss      = traj_loss + 0.1 * conf_loss

        # Skip NaN batches
        if torch.isnan(loss):
            skip_count += 1
            continue

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 0.5)
        optimizer.step()
        train_loss += loss.item()

    scheduler.step()
    avg_loss = train_loss / max(len(train_loader) - skip_count, 1)

    # Validate
    model.eval()
    val_ades, val_fdes = [], []
    with torch.no_grad():
        for obs, pred in val_loader:
            obs  = obs.to(cfg.DEVICE)
            pred = pred.to(cfg.DEVICE)
            trajs, confs = model(obs)
            val_ades.append(compute_ade(trajs, pred))
            val_fdes.append(compute_fde(trajs, pred))

    val_ade = sum(val_ades) / len(val_ades)
    val_fde = sum(val_fdes) / len(val_fdes)
    lr_now  = scheduler.get_last_lr()[0]

    flag = ''
    if val_ade < best_ade:
        best_ade = val_ade
        torch.save({
            'epoch': epoch,
            'state_dict': model.state_dict(),
            'ade': best_ade,
            'fde': val_fde
        }, os.path.join(cfg.SAVE_DIR, 'model_best.pth'))
        flag = '  ** SAVED **'

    elapsed = time.time() - t0
    print(f'Ep {epoch:03d}/{cfg.EPOCHS} | '
          f'loss={avg_loss:.4f} | '
          f'ADE={val_ade:.4f} | '
          f'FDE={val_fde:.4f} | '
          f'lr={lr_now:.2e} | '
          f'{elapsed:.0f}s{flag}')

print(f'\nBest ADE: {best_ade:.4f}')

Resuming from epoch 5, ADE=0.3353
Ep 005/60 | loss=0.1884 | ADE=0.3283 | FDE=0.6040 | lr=9.99e-05 | 57s  ** SAVED **
Ep 006/60 | loss=0.1871 | ADE=0.3212 | FDE=0.5943 | lr=9.97e-05 | 55s  ** SAVED **
Ep 007/60 | loss=0.1865 | ADE=0.3238 | FDE=0.5967 | lr=9.93e-05 | 55s
Ep 008/60 | loss=0.1864 | ADE=0.3268 | FDE=0.6020 | lr=9.88e-05 | 55s
Ep 009/60 | loss=0.1859 | ADE=0.3332 | FDE=0.6118 | lr=9.81e-05 | 56s
Ep 010/60 | loss=0.1857 | ADE=0.3305 | FDE=0.6053 | lr=9.72e-05 | 55s
Ep 011/60 | loss=0.1856 | ADE=0.3232 | FDE=0.5997 | lr=9.62e-05 | 55s
Ep 012/60 | loss=0.1852 | ADE=0.3201 | FDE=0.5933 | lr=9.51e-05 | 55s  ** SAVED **
Ep 013/60 | loss=0.1849 | ADE=0.3302 | FDE=0.6071 | lr=9.38e-05 | 55s
Ep 014/60 | loss=0.1847 | ADE=0.3168 | FDE=0.5840 | lr=9.24e-05 | 56s  ** SAVED **
Ep 015/60 | loss=0.1845 | ADE=0.3177 | FDE=0.5860 | lr=9.09e-05 | 55s
Ep 016/60 | loss=0.1841 | ADE=0.3179 | FDE=0.5883 | lr=8.92e-05 | 55s
Ep 017/60 | loss=0.1840 | ADE=0.3243 | FDE=0.5962 | lr=8.74e-05 | 55s
Ep 0

In [14]:
import torch, os
from torch.amp import autocast

ckpt = torch.load(os.path.join(cfg.SAVE_DIR, 'model_best.pth'))
model.load_state_dict(ckpt['state_dict'])
print(f'Loaded best model from epoch {ckpt["epoch"]}')

model.eval()
test_ades, test_fdes = [], []
with torch.no_grad():
    for obs, pred in test_loader:
        obs  = obs.to(cfg.DEVICE)
        pred = pred.to(cfg.DEVICE)
        trajs, confs = model(obs)
        test_ades.append(compute_ade(trajs, pred))
        test_fdes.append(compute_fde(trajs, pred))

test_ade = sum(test_ades) / len(test_ades)
test_fde = sum(test_fdes) / len(test_fdes)

print('\n=== FINAL TEST RESULTS ===')
print(f'ADE : {test_ade:.4f} m')
print(f'FDE : {test_fde:.4f} m')
print(f'Best epoch : {ckpt["epoch"]}')

Loaded best model from epoch 46

=== FINAL TEST RESULTS ===
ADE : 0.3127 m
FDE : 0.5837 m
Best epoch : 46
